# CISB5123 Text Analytics - Lab Assignment 3
Topic Modeling with LDA using Gensim

1. Muhammad Izzul Danish bin Abdul Rasib 

1. Import the necessary libraries

In [3]:
!pip install gensim

In [4]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

ImportError: cannot import name 'triu' from 'scipy.linalg' (C:\Users\IM11\anaconda3\Lib\site-packages\scipy\linalg\__init__.py)

2. Read the dataset

In [ ]:
df = pd.read_csv('news_dataset.csv', usecols=['text'])
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("Number of null values in text column:", df['text'].isnull().sum())

3. Text pre-processing

The steps below:
- remove null values
- convert text to lowercase
- remove punctuation and non-alphabetic characters
- tokenize text
- remove stopwords
- apply stemming
- apply lemmatization

In [ ]:
df = df.dropna(subset=['text']).copy()
df['text'] = df['text'].astype(str)

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation, numbers, and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords and short tokens
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens

df['processed_text'] = df['text'].apply(preprocess_text)
df.head()

In [ ]:
# Remove empty processed rows if any
df = df[df['processed_text'].map(len) > 0].copy()

print("Number of usable documents after preprocessing:", len(df))
print("Sample processed text:")
print(df['processed_text'].iloc[0][:30]) 

4. Create dictionary and corpus for LDA

In [ ]:
dictionary = corpora.Dictionary(df['processed_text'])

# Filter out very rare and very common words
dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in df['processed_text']]

print("Number of unique tokens:", len(dictionary))
print("Number of documents:", len(corpus))

5. Perform LDA using Gensim (4topics)

In [ ]:
num_topics = 4

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

# Display the discovered topics
for idx, topic in lda_model.print_topics(num_topics=num_topics, num_words=10):
    print(f"Topic {idx + 1}: {topic}")
    print()

6. Evaluate the LDA model using Coherence Score

In [ ]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=df['processed_text'],
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print("Coherence Score:", round(coherence_score, 4))

7. Assign dominant topic to each document

In [ ]:
def get_dominant_topic(bow):
    topic_probs = lda_model.get_document_topics(bow)
    return max(topic_probs, key=lambda x: x[1])[0]

df['dominant_topic'] = [get_dominant_topic(bow) for bow in corpus]

df[['text', 'dominant_topic']].head(10)

8. Interpretation of the coherence score


The coherence score is used to measure how semantically meaningful the topics are based on the words grouped within each topic. In general, a higher coherence score indicates that the generated topics are more interpretable and the top words in each topic are more related to one another. In this assignment, the coherence score obtained from the LDA model suggests that the model produces a reasonably meaningful topic structure for the news dataset. However, the score may still be improved by adjusting the preprocessing steps, tuning the number of topics, or changing the LDA parameters such as passes and alpha.


9. Short conclusion

This notebook successfully performs topic modeling on the `news_dataset.csv` dataset using LDA with Gensim. The text data is preprocessed through stopword removal, stemming, and lemmatization before training the model. Then, the coherence score is calculated to evaluate the quality of the discovered topics. Finally, the topics can be interpreted based on their top keywords and dominant topic assignments.
